# EDA and feature engineering on the selected features

**Purpose.** Explore the 40 columns saved in `outputs/selected_features.json`
and demonstrate, as worked examples, how to engineer features from them. This
notebook is a **guide**, not a pipeline: every engineered feature stays at
notebook level until it earns promotion to `src/daa/features/` (checklist in
§5).

**Provenance.** K=40 shortlist produced by the stage-3 deployed
HistGradientBoosting (baseline macro-F1 0.453 all-recoded; K=40 is the smallest
set that passed the stage-3 rule). The loader below re-checks the column
fingerprint and the target fingerprint, so a stale or hand-edited file fails
loudly instead of silently.

**Standing caveats** (do not drop these when copying cells):

1. `AGE_RANGE` band 0 is disputed (`TODO.md`, open question for the professor).
   Every age interpretation below is provisional; bands are treated as
   unordered categories, not as an ordinal target.
2. `BM1` / `BM2` are height-like and weight-like, but units are unconfirmed
   (`context/competition/data-dictionary.md`).
3. `PA1` and `IN24` were left nominal **on purpose** in `recode.py`. Parsing
   `PA1` here (§3.5) is an experiment with caveats, not a decision.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_selection import mutual_info_classif

from daa.config import OUTPUTS_DIR
from daa.data import TARGET, explicit_spec, infer_column_spec, load_test, load_train
from daa.features.selection import load_selected_features
from daa.splits import load_folds

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
SEED = 42

train_features, target = load_train()
test_features, test_row_ids = load_test()
full_spec = explicit_spec(infer_column_spec(train_features))
selection = load_selected_features(OUTPUTS_DIR / "selected_features.json", full_spec, target)
selected_columns = list(selection.columns)

print(f"k = {selection.k}, column fingerprint {selection.fingerprint[:16]}... verified")
print(f"target fingerprint {selection.target_fingerprint[:16]}... verified")
print("full-frame spec counts:", full_spec.counts())
print(f"train {train_features.shape}, test {test_features.shape}")

## 1. Profile of the 40 selected columns

Split the shortlist into what the recoding made numeric and what stayed
nominal, then profile every column: level count, answered share, dominant
level. The profile decides the EDA plot (distribution for numerics, level
shares for nominals) and flags fragile columns (mostly-sentinel, ultra-rare
levels) before any engineering touches them.

In [ ]:
numeric_columns = [c for c in selected_columns if pd.api.types.is_numeric_dtype(train_features[c])]
nominal_columns = [c for c in selected_columns if c not in numeric_columns]
selected_flags = [c for c in selected_columns if f"{c}_FLAG" in train_features.columns]
print(f"{len(numeric_columns)} numeric, {len(nominal_columns)} nominal")
print(f"{len(selected_flags)} numeric columns have a <COL>_FLAG routing companion: {selected_flags}")

In [ ]:
def profile_column(frame: pd.DataFrame, column: str) -> dict:
    """One profile row for ``column``.

    Purpose: decide how to plot and engineer the column.
    Invariants: never mutates the frame; one pass per column.
    Failure modes: for parsed columns the sentinels were already turned into
    NaN at load time, so ``answer_share`` counts true answers only; for
    nominal columns sentinels are ordinary levels and are kept on purpose
    (who was skipped is signal, see ``recode.py``).
    """
    series = frame[column]
    level_counts = series.value_counts(dropna=False)
    top_level, top_count = level_counts.index[0], level_counts.iloc[0]
    return {
        "dtype": str(series.dtype),
        "n_levels": series.nunique(dropna=False),
        "answered": int(series.notna().sum()),
        "answer_%": round(series.notna().mean() * 100, 1),
        "top_level": str(top_level)[:40],
        "top_%": round(100 * top_count / len(series), 1),
    }


profile = pd.DataFrame(
    {column: profile_column(train_features, column) for column in selected_columns}
).T
display(profile)

How to read the profile (all numbers measured on the current `data/`):

- The household block (`HH_INACT`, `HH_ACT`, `HHNBPERS_COD`,
  `HHNBPERS_0_13_COD`) is always answered: it describes the household, not
  the person, so it is never routed away.
- Several selected columns are **mostly sentinel**: `IN4` is answered on ~7%
  of rows, `IN45` on ~20%, `IN47` on ~30%. Their value is partly in *whether*
  they were asked; the `*_FLAG` columns keep that (§2.4).
- Several nominals are routing-dominated: `FT_PT`, `JOBSTAT_COD`, `NT1`,
  `PC2`, `IN22`, `PA7_NAC` have `'Não aplicável'` as the top level, at 42-92%
  of rows. Beware one-hot encoding that 'takes' the top level as the normal
  case: what it really encodes is *being outside the survey path*, which is
  strongly age-related.
- `MAINSTAT` has one level with a single row ("presta serviço cívico").
  Rare-level grouping is a **learned** transform and belongs in
  `preprocess.py` inside the pipeline, never in a notebook cell.

## 2. EDA against the target

Everything in this section is measured on the **full training frame**, so it
is descriptive: it motivates candidate features, it does not validate them.
Validation happens in §4 on the frozen folds.

Two deliberate choices:

- Bands are treated as **unordered categories**. We do not correlate anything
  against the band number, because the band-0 meaning is open (`TODO.md`) and
  ordinal treatment of the target is blocked until the professor answers.
- Plots show all 16 bands; interpret the *shape across bands*, not single-band
  differences (the smallest band has 172 rows).

In [ ]:
medians_by_band = train_features[numeric_columns].groupby(target).median()
display(medians_by_band.T.round(1).rename(columns=lambda b: f"band {b}"))

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 13), sharex=True)
band_labels = sorted(target.unique())
for column, ax in zip(numeric_columns, axes.ravel()):
    values_by_band = [
        train_features.loc[target == band, column].dropna().to_numpy()
        for band in band_labels
    ]
    ax.boxplot(values_by_band, showfliers=False)
    ax.set_xticks(range(1, len(band_labels) + 1), band_labels, fontsize=7)
    ax.set_title(column, fontsize=10)
    ax.tick_params(axis="y", labelsize=7)
fig.suptitle("Numeric selected columns by AGE_RANGE band (median, IQR; provisional band labels)", y=1.0)
fig.tight_layout()
plt.show()

### 2.1 Redundancy between the numeric columns

If two selected columns carry almost the same information, an engineered
combination (a ratio, a difference) may say something neither says alone, and
a linear model will happily trade one for the other. Look for pairs with
|Spearman| well above 0.8: they are candidates for combination or removal.

In [ ]:
numeric_spearman = train_features[numeric_columns].corr(method="spearman")

upper_triangle_mask = np.triu(np.ones(numeric_spearman.shape, dtype=bool), k=1)
redundant_pairs = (
    numeric_spearman.where(upper_triangle_mask)
    .stack()
    .abs()
    .sort_values(ascending=False)
    .head(8)
)
display(redundant_pairs.to_frame(name="|spearman|"))

fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(numeric_spearman.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_columns)), numeric_columns, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(numeric_columns)), numeric_columns, fontsize=8)
for row in range(len(numeric_columns)):
    for col in range(len(numeric_columns)):
        ax.text(col, row, f"{numeric_spearman.iloc[row, col]:.2f}",
                ha="center", va="center", fontsize=6,
                color="black" if abs(numeric_spearman.iloc[row, col]) < 0.6 else "white")
fig.colorbar(image, ax=ax, shrink=0.8)
ax.set_title("Spearman correlation between the numeric selected columns")
fig.tight_layout()
plt.show()

### 2.2 Nominal level shares inside each band

For each key nominal column, the heatmap shows which levels characterise
which bands: every panel row is one band, rows sum to 100%. Included levels
keep `'Não aplicável'` on purpose: the *share of non-applicable inside a band*
is often the strongest age signal in the panel.

In [ ]:
focus_nominals = ["MAINSTAT", "HATLEVEL", "MARSTALEGAL", "HHTYPE", "FT_PT"]


def level_shares_by_band(frame: pd.DataFrame, target_series: pd.Series, column: str) -> pd.DataFrame:
    """Percent share of each level of ``column`` inside every target band.

    Purpose: see which levels characterise which age bands.
    Invariants: each row (band) sums to 100; the frame is not mutated; levels
    unseen in ``frame`` simply do not appear.
    Failure modes: sentinel levels are kept on purpose (routing is signal, see
    ``recode.py``); with rare levels the shares are noisy because the small
    bands have 172-450 rows.
    """
    counts = pd.crosstab(target_series, frame[column])
    return counts.div(counts.sum(axis=1), axis=0) * 100.0


fig, axes = plt.subplots(len(focus_nominals), 1, figsize=(12, 3.4 * len(focus_nominals)))
for column, ax in zip(focus_nominals, axes.ravel()):
    shares = level_shares_by_band(train_features, target, column)
    ax.imshow(shares.to_numpy(), aspect="auto", cmap="Blues", vmin=0, vmax=100)
    ax.set_xticks(range(len(shares.columns)),
                  [str(level)[:24] for level in shares.columns],
                  rotation=30, ha="right", fontsize=7)
    ax.set_yticks(range(len(shares.index)), shares.index, fontsize=7)
    for j, level in enumerate(shares.columns):
        for i, band in enumerate(shares.index):
            ax.text(j, i, f"{shares.iloc[i, j]:.0f}", ha="center", va="center",
                    fontsize=6, color="black" if shares.iloc[i, j] < 55 else "white")
    ax.set_title(f"{column}: level share within each band (%)", fontsize=11)
fig.tight_layout()
plt.show()

### 2.3 The routing flags across bands

The `*_FLAG` companions record, per parsed column, whether the person was
asked: `"answer"`, `'Sem resposta'`, or `'Não aplicável'`. The survey routes
by age (students are not asked about retirement), so the **answer rate is
itself an age signal** — this is the quantitative version of what §1 flagged
qualitatively.

In [ ]:
flag_answer_rate = pd.DataFrame(
    {column: train_features[f"{column}_FLAG"].eq("answer").groupby(target).mean() * 100
     for column in selected_flags}
)

fig, ax = plt.subplots(figsize=(13, 5))
ax.imshow(flag_answer_rate.to_numpy().T, aspect="auto", cmap="Greens", vmin=0, vmax=100)
ax.set_xticks(range(len(flag_answer_rate.index)), flag_answer_rate.index, fontsize=8)
ax.set_yticks(range(len(flag_answer_rate.columns)), flag_answer_rate.columns, fontsize=8)
ax.set_xlabel("AGE_RANGE band (provisional labels)")
for i, column in enumerate(flag_answer_rate.columns):
    for j, band in enumerate(flag_answer_rate.index):
        ax.text(j, i, f"{flag_answer_rate.iloc[j, i]:.0f}", ha="center", va="center",
                fontsize=6, color="black" if flag_answer_rate.iloc[j, i] < 55 else "white")
ax.set_title("Answer rate (%) of each routed selected column, by band")
fig.tight_layout()
plt.show()

## 3. Feature engineering, by example

Every example below is one of three kinds, and the kind decides where it may
live:

| Kind | Definition | Where it may live | Examples |
|---|---|---|---|
| Rule-book | fixed value-to-value map, nothing estimated from the rows | load time, like `recode_frame`; future `src/daa/features/engineer.py` | BMI, children ratio |
| Learned | estimates something from the rows (a median, a category frequency) | **only** inside the pipeline in `preprocess.py`, refit per CV fold | rare-level grouping, target encoding |
| Parse experiment | rule-book, but the column's meaning is unconfirmed | notebook only, until the codebook confirms | `PA1` months-since |

The dividing line is leakage (see the `recode.py` docstring): a transform
leaks when it *estimates* from rows it must not see. A rule-book transform
cannot leak, so it may run at load time. Each function below states its
purpose, invariants, and failure modes — copy that pattern.

### 3.1 BMI: combine two correlated raw columns

`BM1` and `BM2` are height-like and weight-like (measured ranges 130-205 and
32-150; 10,559 of 10,962 rows have both). A model can learn the combination,
but tree models split axis-aligned: one scale-free combination helps them ask
a single question. Units are unconfirmed, so read the result as a ratio.

In [ ]:
def bmi_feature(frame: pd.DataFrame) -> pd.Series:
    """Body-mass-index-like ratio from BM1 (height-like) and BM2 (mass-like).

    Purpose: one scale-free feature instead of two correlated raw columns.
    Invariants: pure, never mutates the frame; missing BM1 or BM2 gives NaN;
    works on any frame that has the two columns (train or test).
    Failure modes: a height of 0 or less returns NaN instead of exploding;
    units of BM1/BM2 are unconfirmed, so the report must call this a ratio
    until the codebook confirms cm/kg.
    """
    height_metres = frame["BM1"] / 100.0
    mass = frame["BM2"]
    bmi = mass / height_metres**2
    return bmi.where(height_metres > 0).rename("BMI")


bmi_train = bmi_feature(train_features)
bmi_test = bmi_feature(test_features)
print("train:", bmi_train.describe().round(2).to_dict())
print("test :", bmi_test.describe().round(2).to_dict())

fig, ax = plt.subplots(figsize=(10, 4))
ax.boxplot(
    [bmi_train[target == band].dropna() for band in sorted(target.unique())],
    showfliers=False,
)
ax.set_xticks(range(1, 17), sorted(target.unique()), fontsize=8)
ax.set_xlabel("AGE_RANGE band (provisional labels)")
ax.set_ylabel("BMI-like ratio")
ax.set_title("BMI by band — a single question a tree can split on")
fig.tight_layout()
plt.show()

### 3.2 Children per household member: a ratio of two always-answered columns

Both `HHNBPERS_COD` and `HHNBPERS_0_13_COD` are always answered and floored
buckets at the top (`'>=7'` becomes 7, `'>=3'` becomes 3, see `recode.py`), so
the ratio is approximate at the top of the scale. Note the standing limit:
`HHID` is unique in the training frame, so **household-level aggregation
features are impossible** (no two rows share a household); per-person ratios
are all we can build.

In [ ]:
def children_ratio_feature(frame: pd.DataFrame) -> pd.Series:
    """Children 0-13 per household member.

    Purpose: one dependency-load feature from two always-answered columns.
    Invariants: pure, never mutates the frame; missing either column gives
    NaN; the bucket floors make the value approximate, never exact.
    Failure modes: a zero or missing household size gives NaN instead of a
    division error (measured today the minimum is 1, but the guard keeps the
    rule safe if a future file breaks that).
    """
    household_size = frame["HHNBPERS_COD"]
    children = frame["HHNBPERS_0_13_COD"]
    ratio = children / household_size
    return ratio.where(household_size > 0).rename("CHILDREN_RATIO")


children_ratio_train = children_ratio_feature(train_features)
children_ratio_test = children_ratio_feature(test_features)
display(
    pd.DataFrame(
        {
            "train": children_ratio_train.describe().round(3),
            "test": children_ratio_test.describe().round(3),
        }
    )
)

### 3.3 Binary indicators from `MAINSTAT` (and one interaction)

`MAINSTAT` has 9 long Portuguese level strings. Two cautions:

- The functions below depend on the **exact survey wording** the way
  `recode.py` depends on the sentinel spellings: a relabel upstream silently
  turns a level into `False`. That is a parse contract, not a model choice.
- The last level has one row. Do not group it here: rare-level grouping is
  learned, so it belongs in `preprocess.py`.

The interaction at the end demonstrates the pattern; keep interactions only
if §4 (or a stage-4 run) shows they earn their place. Trees find most
interactions themselves; explicit interactions mainly help linear models.

In [ ]:
def activity_indicators(frame: pd.DataFrame) -> pd.DataFrame:
    """Three binary indicators derived from MAINSTAT levels.

    Purpose: give models direct, cheap access to the life-stage facts that
    dominate the MAINSTAT panel in §2.2.
    Invariants: pure, never mutates the frame; an unknown level yields all
    three indicators False, never an error.
    Failure modes: string matching on exact survey wording — if the upstream
    label changes, the level silently becomes False. This is a parse
    contract; verify against the data dictionary when labels are edited.
    """
    main_status = frame["MAINSTAT"]
    return pd.DataFrame(
        {
            "IS_EMPLOYED": main_status.str.startswith("Tem um emprego").fillna(False),
            "IS_RETIRED": main_status.str.startswith("Reformado").fillna(False),
            "IS_STUDENT": main_status.str.startswith("Aluno").fillna(False),
        },
        index=frame.index,
    )


indicators_train = activity_indicators(train_features)
indicators_test = activity_indicators(test_features)
for column in indicators_train.columns:
    share = indicators_train[column].mean() * 100
    print(f"{column}: {share:.1f}% in train, {indicators_test[column].mean() * 100:.1f}% in test")

# Example interaction: retired AND married. A rule-book AND of two rules.
retired_and_married = (
    indicators_train["IS_RETIRED"] & train_features["MARSTALEGAL"].eq("Casado")
).rename("RETIRED_AND_MARRIED")
display(retired_and_married.value_counts().to_frame("rows"))

### 3.4 One answered-count for the whole routed block

§2.3 showed the answer rates move with the band, one column at a time. The
loose end in the `recode.py` docstring suggests collapsing the flag block for
models that struggle with 29 one-hot flags. This goes the other way for the
guide: **sum** the flags over every parsed column (not just the selected ones)
into one count. It is a rule-book sum, so it may run at load time; whether one
count beats 29 flags is a stage-4 question, not a notebook conclusion.

In [ ]:
def answered_count_feature(frame: pd.DataFrame) -> pd.Series:
    """Number of parsed quantity questions this person answered.

    Purpose: collapse the 29 routing flags into one 'how many questions
    applied to this person' signal.
    Invariants: pure, never mutates the frame; counts only 'answer' flags;
    formed over the whole 29-column parsed block, not just the selected ones,
    so it summarises the survey path, not the shortlist.
    Failure modes: on a frame without flag columns (raw CSV read directly,
    bypassing load_train/load_test) the count is all zeros — the flags exist
    only after recoding.
    """
    flag_columns = [c for c in frame.columns if c.endswith("_FLAG")]
    return frame[flag_columns].eq("answer").sum(axis=1).rename("ANSWERED_COUNT")


answered_train = answered_count_feature(train_features)
answered_test = answered_count_feature(test_features)
display(pd.DataFrame({"train": answered_train.describe().round(2),
                       "test": answered_test.describe().round(2)}))
display(answered_train.groupby(target).mean().round(2).to_frame("ANSWERED_COUNT mean (train)"))

### 3.5 Parse experiment: `PA1` from text to months

`PA1` mixes `'Nunca'` (56.7%), `'Há mais de 1 ano'` (20.1%), 6-digit `YYYYMM`
values (22.6%), a year-only `'2018'` (51 rows), and `'Sem resposta'`.
`recode.py` left it nominal on purpose. This cell demonstrates how a parse
*hypothesis* is built and checked — it is **not** promoted, and it must not
be copied into `recode.py` until the codebook confirms what `PA1` asks.

Two deliberate design points:

- The elapsed time uses `REFYEAR`/`REFMONTH`. Those columns are admin
  metadata dropped by `explicit_spec`: a feature may *use* them as
  ingredients, they just do not reach the model as inputs themselves.
- The 51 year-only `'2018'` rows are excluded (the month is unknown), and
  negative elapsed values are clamped to 0: a visit in the interview month
  is 'this month', not a time machine.

In [ ]:
def months_since_visit_feature(frame: pd.DataFrame) -> pd.Series:
    """Months between the PA1 'YYYYMM' answer and the interview date.

    Purpose: EXPERIMENT - demonstrate a mixed-text parse with explicit
    fallbacks for every non-timestamp level.
    Invariants: pure, never mutates the frame; only exact 6-digit YYYYMM
    values produce a number; 'Nunca', 'Há mais de 1 ano', 'Sem resposta',
    year-only '2018' and NaN all become NaN; negative elapsed time (answer
    after the interview month) is clamped to 0.
    Failure modes: depends on REFYEAR/REFMONTH being present in the frame
    (they are, at load time; explicit_spec only drops them at spec level);
    assumes but does not confirm that YYYYMM is 'when did you last ...' -
    the codebook question for PA1 is still open.
    """
    text = frame["PA1"].astype("string")
    timestamp_mask = text.str.fullmatch(r"\d{6}").fillna(False)
    visit_year = pd.to_numeric(text.where(timestamp_mask).str.slice(0, 4), errors="coerce")
    visit_month = pd.to_numeric(text.where(timestamp_mask).str.slice(4, 6), errors="coerce")
    elapsed_months = (frame["REFYEAR"] - visit_year) * 12 + (frame["REFMONTH"] - visit_month)
    return elapsed_months.clip(lower=0).rename("MONTHS_SINCE_VISIT")


months_train = months_since_visit_feature(train_features)
months_test = months_since_visit_feature(test_features)
print(f"parsed rows: train {months_train.notna().sum()}, test {months_test.notna().sum()}")
display(months_train[months_train.notna()].value_counts().sort_index().head(24).to_frame("rows"))

fig, ax = plt.subplots(figsize=(10, 4))
valid_months = months_train[target.notna()]
ax.boxplot([valid_months[target == band].dropna() for band in sorted(target.unique())], showfliers=False)
ax.set_xticks(range(1, 17), sorted(target.unique()), fontsize=8)
ax.set_xlabel("AGE_RANGE band (provisional labels)")
ax.set_ylabel("months since PA1 answer")
ax.set_title("MONTHS_SINCE_VISIT by band — interpretation deferred until the codebook answers")
fig.tight_layout()
plt.show()

### 3.6 What never goes in a notebook cell

Anything that **learns from the rows** lives in `preprocess.py`, inside the
pipeline the CV fits, so each fold refits it on training rows only:

| Transform | Example | Correct home |
|---|---|---|
| Imputation from data | median of a column | pipeline `SimpleImputer` |
| Scaling | standardisation | pipeline `StandardScaler` |
| Rare-level grouping | collapse MAINSTAT's 1-row level | `preprocess.py`, refit per fold |
| Target encoding | any band-conditional statistic | `preprocess.py` or custom transformer |

Doing any of these on the full frame before the split is the leakage bug the
primer notebook (`00_primer.ipynb`) demonstrates.

## 4. A light out-of-fold check of the engineered features

Descriptive (§2) is not validated. The cheapest honest check available:

- Take the **frozen folds** (`splits/folds.npz`, 5 splits x 3 repeats, seed
  42, same folds every stage-4 run uses).
- On each fold, compute the **mutual information** between the feature and
  the target **using training rows only**; report the mean and the spread
  across the 15 folds.

MI is a *screen*, not a selector (the stage-3 lesson): it prunes obviously
dead features and ranks candidates; only a full stage-4 run with the frozen
folds decides whether a feature stays. Parents sit in the same table so each
engineered feature is compared against what it came from.

In [ ]:
candidate_features: dict[str, tuple[pd.Series, str]] = {
    # engineered, rule-book
    "BMI": (bmi_feature(train_features), "numeric"),
    "CHILDREN_RATIO": (children_ratio_feature(train_features), "numeric"),
    "ANSWERED_COUNT": (answered_count_feature(train_features), "numeric"),
    "MONTHS_SINCE_VISIT": (months_since_visit_feature(train_features), "numeric"),
    "IS_RETIRED": (indicators_train["IS_RETIRED"].astype("float64"), "discrete"),
    "RETIRED_AND_MARRIED": (retired_and_married.astype("float64"), "discrete"),
    # parents, for comparison
    "BM1": (train_features["BM1"], "numeric"),
    "BM2": (train_features["BM2"], "numeric"),
    "PE9": (train_features["PE9"], "numeric"),
    "HHNBPERS_COD": (train_features["HHNBPERS_COD"], "numeric"),
    "HHNBPERS_0_13_COD": (train_features["HHNBPERS_0_13_COD"], "numeric"),
    "MAINSTAT": (train_features["MAINSTAT"], "discrete"),
}
folds = load_folds(target)
print(f"{len(folds)} folds loaded (5 splits x 3 repeats, frozen)")


def out_of_fold_mi(feature: pd.Series, kind: str, folds) -> dict[str, float]:
    """MI between feature and target, averaged over the frozen folds.

    Purpose: a leakage-free screen of a candidate feature; validation rows
    never touch the estimate.
    Invariants: computed on fold.train_idx rows only; random_state fixed at
    SEED so the table is reproducible; rows where the feature is missing or
    unmatched are dropped pairwise, selected the same way in every fold.
    Failure modes: nominal features are factorized, so MI understates their
    information (codes entangle rare levels); a feature present only on a
    routing subset (like MONTHS_SINCE_VISIT) is screened on that subset -
    a fair read compares it with its parent, not with always-answered
    columns.
    """
    target_values = target.to_numpy(dtype="int64")
    if kind == "numeric":
        values = feature.to_numpy(dtype="float64")
        valid_rows = np.isfinite(values)
        matrix = values.reshape(-1, 1)
        discrete = False
    else:
        codes, _ = pd.factorize(feature.fillna("(no level)"))
        values = codes.astype("float64")
        valid_rows = codes >= 0
        matrix = values.reshape(-1, 1)
        discrete = True
    fold_mi = [
        mutual_info_classif(
            matrix[np.intersect1d(fold.train_idx, np.flatnonzero(valid_rows))],
            target_values[np.intersect1d(fold.train_idx, np.flatnonzero(valid_rows))],
            discrete_features=discrete,
            random_state=SEED,
        )[0]
        for fold in folds
    ]
    return {"mi_mean": float(np.mean(fold_mi)), "mi_std": float(np.std(fold_mi)),
            "n_folds": len(fold_mi)}


mi_table = pd.DataFrame(
    {name: {"kind": kind, **out_of_fold_mi(series, kind, folds)}
     for name, (series, kind) in candidate_features.items()}
).T.sort_values("mi_mean", ascending=False)
display(mi_table.round(4))

Reading the table:

- Higher `mi_mean` = the feature alone answers more about the band;
  `mi_std` = instability across folds (a large spread is a warning even when
  the mean looks good).
- The useful comparisons are **child vs parents**: `BMI` against `BM1` and
  `BM2` together, `CHILDREN_RATIO` against its two parents,
  `MONTHS_SINCE_VISIT` against the raw `PA1` (absent from the table because
  as text its factorized codes say almost nothing — which is exactly why
  the parse experiment was worth running).
- The screen cannot see interactions with other features or pruning between
  correlated columns. A feature that screens poorly may still earn its place
  in a model; a feature that screens well still has to win a stage-4 run.

## 5. Checklist: what to do with each example

| Feature | Kind | Verdict | Next action |
|---|---|---|---|
| `BMI` | rule-book | promote candidate | add to a future `src/daa/features/engineer.py` (pure, load-time, same contract as `recode_frame`); confirm units at the codebook before the report |
| `CHILDREN_RATIO` | rule-book | promote candidate | same module; document the bucket-floor approximation in its docstring |
| `ACTIVITY_INDICATORS` | rule-book | keep, with a test | the exact-string contract needs a unit test against the survey wording before promotion |
| `ANSWERED_COUNT` | rule-book | try in stage-4 | cheap, zero-risk to compute; only a stage-4 run can tell if one count beats 29 flags |
| `MONTHS_SINCE_VISIT` | parse experiment | wait | blocked on the `PA1` codebook question; if confirmed, the parse moves into `recode.py` (today the guard would fail it) |
| `RETIRED_AND_MARRIED` | interaction | undecided | only keep if a stage-4 run beats the plain flags; trees rarely need it |

Promotion rules, for anything above:

1. The transform lands in `src/daa/features/engineer.py` as a pure,
   stateless function over a frame (the `recode_frame` contract), plus a
   small test in `tests/`.
2. The decision comes from a stage-4 run on the frozen folds — never from
   this notebook's tables.
3. Anything learned from the rows goes to `preprocess.py` inside the
   pipeline instead, no exceptions.

Standing caveats that travel with all of these: band 0 is disputed
(`TODO.md`), `BM1`/`BM2` units are unconfirmed, and household aggregation is
impossible (`HHID` is unique in train).